# Integración Consumo + Resistencia (2000–2024)

**Objetivo:** enriquecer el dataset de **resistencia antimicrobiana** con el dataset de **consumo de antibióticos** mediante un **LEFT JOIN**.

- **Dataset izquierda (se mantiene completo):** `antimicrobial_resistance_2000_2024_processed.csv`
- **Dataset derecha (enriquece):** `antibiotic_consumption_2000_2024_processed.csv`
- **Claves de unión (join keys):**
  - `country`
  - `year`
  - Grupo **ATC**: `atc_group` (resistencia) = `antibiotic_group` (consumo)

**Salida final:** `../data/processed/consumption_resistance_2000_2024_processed.csv` con columnas:
1. Country
2. Year
3. Bacteria
4. Antibiotic group *(mapeado a nombre legible)*
5. Resistance
6. Consumption


## 1) Imports
Cargamos las librerías necesarias.

In [2]:
import pandas as pd

## 2) Carga de datos
Leemos los CSV procesados desde la carpeta `../data/processed/`.

In [3]:
# Rutas de entrada
PATH_RESISTANCE = "../data/processed/antimicrobial_resistance_2000_2024_processed.csv"
PATH_CONSUMPTION = "../data/processed/antibiotic_consumption_2000_2024_processed.csv"

# Lectura de datos
df_res = pd.read_csv(PATH_RESISTANCE)
df_con = pd.read_csv(PATH_CONSUMPTION)

# Vista rápida
print("✅ CSVs cargados")
print("Resistencia:", df_res.shape)
print("Consumo:", df_con.shape)

✅ CSVs cargados
Resistencia: (6161, 7)
Consumo: (1725, 4)


## 3) Revisión rápida de columnas
Comprobamos que existen las columnas clave antes del merge.

In [4]:
print("Columnas RESISTENCIA:")
print(df_res.columns)

print("\nColumnas CONSUMO:")
print(df_con.columns)

Columnas RESISTENCIA:
Index(['year', 'country_code', 'country', 'bacteria', 'antibiotic_group',
       'resistance_percentage', 'atc_group'],
      dtype='object')

Columnas CONSUMO:
Index(['year', 'country', 'antibiotic_group',
       'ddd_per_1000_inhabitants_per_day'],
      dtype='object')


## 4) Selección de columnas en consumo
Nos quedamos solo con lo necesario para el merge y el consumo final.

In [5]:
# Nos quedamos solo con columnas necesarias del dataset de consumo
df_con = df_con[[
    "country",
    "year",
    "antibiotic_group",
    "ddd_per_1000_inhabitants_per_day"
]].copy()

df_con.head()

,country,year,antibiotic_group,ddd_per_1000_inhabitants_per_day
0,Belgium,2000,J01D,4.3713
1,Bulgaria,2000,J01D,2.5258
2,Croatia,2000,J01D,3.7358
3,Denmark,2000,J01D,0.1957
4,Finland,2000,J01D,3.0572


## 5) LEFT JOIN (Resistencia ⟵ Consumo)
Hacemos un **LEFT JOIN** para mantener todas las filas de resistencia y añadir consumo cuando haya match.

In [7]:
# LEFT JOIN:
# - Se mantienen TODAS las filas de df_res
# - df_con solo enriquece cuando coincide por país, año y grupo ATC
df_merged = df_res.merge(
    df_con,
    how="left",
    left_on=["country", "year", "atc_group"],
    right_on=["country", "year", "antibiotic_group"]
)

df_merged.head()

,year,country_code,country,bacteria,antibiotic_group_x,resistance_percentage,atc_group,antibiotic_group_y,ddd_per_1000_inhabitants_per_day
0,2012,BG,Bulgaria,Acinetobacter spp.,Aminoglycosides,58.461538,J01G,J01G,0.2826
1,2012,CY,Cyprus,Acinetobacter spp.,Aminoglycosides,52.173913,J01G,J01G,0.0776
2,2012,DE,Germany,Acinetobacter spp.,Aminoglycosides,5.882353,J01G,NaN,NaN
3,2012,DK,Denmark,Acinetobacter spp.,Aminoglycosides,10.389610,J01G,J01G,0.0585
4,2012,EL,Greece,Acinetobacter spp.,Aminoglycosides,78.119935,J01G,J01G,0.1423


## 6) Validación: no perder filas
Confirmamos que el merge no ha eliminado registros del dataset de resistencia.

In [8]:
print("Filas resistencia:", df_res.shape[0])
print("Filas merged:", df_merged.shape[0])

assert df_res.shape[0] == df_merged.shape[0], "⚠️ Se han perdido filas: revisa las claves del merge."

print("✅ Validación OK: no se han perdido filas.")

Filas resistencia: 6161
Filas merged: 6161
✅ Validación OK: no se han perdido filas.


## 7) Dataset final: columnas y nombres
Construimos el dataframe final con el orden y nombres exactos solicitados.

In [9]:
# Selección de columnas finales (antes de renombrar)
df_final = df_merged[[
    "country",
    "year",
    "bacteria",
    "atc_group",
    "resistance_percentage",
    "ddd_per_1000_inhabitants_per_day"
]].copy()

# Renombrado para entrega final
df_final = df_final.rename(columns={
    "country": "Country",
    "year": "Year",
    "bacteria": "Bacteria",
    "atc_group": "Antibiotic group",
    "resistance_percentage": "Resistance",
    "ddd_per_1000_inhabitants_per_day": "Consumption"
})

df_final.head()

,Country,Year,Bacteria,Antibiotic group,Resistance,Consumption
0,Bulgaria,2012,Acinetobacter spp.,J01G,58.461538,0.2826
1,Cyprus,2012,Acinetobacter spp.,J01G,52.173913,0.0776
2,Germany,2012,Acinetobacter spp.,J01G,5.882353,NaN
3,Denmark,2012,Acinetobacter spp.,J01G,10.389610,0.0585
4,Greece,2012,Acinetobacter spp.,J01G,78.119935,0.1423


## 8) Mapeo de ATC → nombre legible
Convertimos los códigos ATC a nombres más entendibles para la presentación.

**Mapeo**
- `J01G` → Aminoglucósidos
- `J01B` → β-lactámicos
- `J01M` → Fluoroquinolonas

Si aparece un ATC no contemplado, se mantiene tal cual.

In [10]:
def map_atc_to_label(series: pd.Series) -> pd.Series:
    """
    Mapea códigos ATC a etiquetas legibles.

    Parameters
    ----------
    series : pd.Series
        Serie con códigos ATC (por ejemplo: 'J01G', 'J01B', 'J01M').

    Returns
    -------
    pd.Series
        Serie con las etiquetas mapeadas. Los valores no incluidos en el mapeo
        se conservan sin cambios.
    """
    mapping = {
        "J01G": "Aminoglucósidos",
        "J01B": "β-lactámicos",
        "J01M": "Fluoroquinolonas",
    }
    return series.map(mapping).fillna(series)


df_final["Antibiotic group"] = map_atc_to_label(df_final["Antibiotic group"])

df_final["Antibiotic group"].value_counts(dropna=False).head(10)

Antibiotic group
J01D                2872
Fluoroquinolonas    1645
Aminoglucósidos     1644
Name: count, dtype: int64

## 9) Exportación
Guardamos el CSV final en la ruta indicada.

In [11]:
OUTPUT_PATH = "../data/processed/consumption_resistance_2000_2024_processed.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print("✅ Archivo guardado en:", OUTPUT_PATH)
print("Shape final:", df_final.shape)

✅ Archivo guardado en: ../data/processed/consumption_resistance_2000_2024_processed.csv
Shape final: (6161, 6)


## 10) Control de calidad rápido
Vemos cuántas filas han quedado sin consumo (NaN) tras el merge, para interpretar cobertura de datos.

In [12]:
total_rows = df_final.shape[0]
missing_consumption = df_final["Consumption"].isna().sum()
pct_missing = (missing_consumption / total_rows) * 100

print(f"Filas totales: {total_rows}")
print(f"Filas SIN consumo (NaN): {missing_consumption} ({pct_missing:.2f}%)")

Filas totales: 6161
Filas SIN consumo (NaN): 1095 (17.77%)
